# 🤖 Lexbot - RAG Chatbot Pháp Lý Việt Nam
**Model:** Qwen2.5-7B-Instruct + QLoRA fine-tuning
**Database:** Neo4j Graph DB
**Data:** Bộ luật Hình sự 2025 + Nghị định Giao thông

In [ ]:
# ─── 1. Cài đặt thư viện ───────────────────────────────────────────────────────
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.9.6 \
    accelerate==0.33.0 bitsandbytes==0.43.3 \
    sentence-transformers datasets loguru \
    neo4j langchain langchain-community

import subprocess
subprocess.run(['nvidia-smi'], check=True)
print('✅ Libraries installed!')

In [ ]:
# ─── 2. Clone / Upload code ────────────────────────────────────────────────────
import os, sys

# Option A: Upload trực tiếp từ Kaggle Dataset
# PROJECT_DIR = '/kaggle/input/lexgpt-qwen'

# Option B: Clone từ GitHub
# !git clone https://github.com/your-username/lexgpt_qwen.git /kaggle/working/lexgpt_qwen
# PROJECT_DIR = '/kaggle/working/lexgpt_qwen'

# Option C: Dùng thư mục hiện tại (khi upload notebook + code cùng dataset)
PROJECT_DIR = '/kaggle/working'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ─── 3. Copy dữ liệu CSV ───────────────────────────────────────────────────────
import shutil
from pathlib import Path

raw_dir = Path(PROJECT_DIR) / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

# Copy từ Kaggle Dataset input
for csv_file in Path('/kaggle/input').rglob('*.csv'):
    dest = raw_dir / csv_file.name
    shutil.copy(csv_file, dest)
    print(f'Copied: {csv_file.name}')

print('Files in raw_dir:', list(raw_dir.iterdir()))

In [ ]:
# ─── 4. Tiền xử lý dữ liệu ────────────────────────────────────────────────────
from data.preprocess import preprocess_all

records = preprocess_all()
print(f'Total records: {len(records)}')
print('Sample:', records[0])

In [ ]:
# ─── 5. Xây dựng Dataset ──────────────────────────────────────────────────────
from dataset.build_dataset import build_and_save

train_ds, val_ds = build_and_save()
print(f'Train: {len(train_ds)} samples')
print(f'Val: {len(val_ds)} samples')
print('\nSample entry:')
print(train_ds[0]['text'][:500])

In [ ]:
# ─── 6. Fine-tune Qwen2.5-7B ──────────────────────────────────────────────────
from trainer.finetune import train, ModelConfig, DataConfig, KaggleTrainConfig

HF_TOKEN = 'hf_your_token_here'  # Thay bằng token của bạn
HUB_MODEL_ID = 'your-username/lexbot-qwen2.5-7b'  # Tùy chọn

model_cfg = ModelConfig(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    use_4bit=True,
    lora_r=16,
    lora_alpha=32,
    max_seq_length=2048,
)
data_cfg = DataConfig(
    dataset_path='/kaggle/working/dataset',
    output_dir='/kaggle/working/lexbot-qwen2.5-7b',
)
train_cfg = KaggleTrainConfig(
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    push_to_hub=False,  # Đặt True để upload lên HuggingFace
    hub_model_id=HUB_MODEL_ID,
)

train(model_cfg, data_cfg, train_cfg)

In [ ]:
# ─── 7. Test inference (không cần Neo4j) ──────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

MODEL_DIR = '/kaggle/working/lexbot-qwen2.5-7b'

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)

def ask_lexbot(question):
    messages = [
        {"role": "system", "content": "Bạn là Lexbot - trợ lý pháp lý Việt Nam."},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Test
answer = ask_lexbot('Điều 134 Bộ luật Hình sự quy định về tội gì?')
print('Lexbot:', answer)

In [ ]:
# ─── 8. Lưu model ra Kaggle Output ────────────────────────────────────────────
# Model đã được lưu tại /kaggle/working/lexbot-qwen2.5-7b
# Bạn có thể tải về hoặc push lên HuggingFace Hub

import os
for f in os.listdir('/kaggle/working/lexbot-qwen2.5-7b'):
    size = os.path.getsize(f'/kaggle/working/lexbot-qwen2.5-7b/{f}')
    print(f'{f}: {size/1e6:.1f} MB')

print('\n✅ Training hoàn tất! Model sẵn sàng sử dụng.')